### ライブラリのインストール


In [1]:
!pip install --upgrade langchain langchain-openai langgraph pydantic python-dotenv faiss-cpu

  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached pydantic_core-2.33.2-cp310-cp310-win_amd64.whl.metadata (6.9 kB)
  Using cached xxhash-3.5.0-cp310-cp310-win_amd64.whl.metadata (13 kB)
Using cached langchain-0.3.27-py3-none-any.whl (1.0 MB)
Using cached pydantic_core-2.33.2-cp310-cp310-win_amd64.whl (2.0 MB)
   ---------------------------------------- 0.0/946.9 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/946.9 kB ? eta -:--:--
   --------------------- ---------------- 524.3/946.9 kB 840.2 kB/s eta 0:00:01
   ---------------------------------------- 946.9/946.9 kB 1.5 MB/s  0:00:00
Using cached xxhash-3.5.0-cp310-cp310-win_amd64.whl (30 kB)

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.27.2

    Uninstalling pydantic_core-2.27.2:

      Successfully uninstalled pydantic_core-2.27.2

   ----- ----------------------------------  2/15 [pydantic-core]
   ----- ---------------------------------

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.0 requires langsmith<0.2.0,>=0.1.112, but you have langsmith 0.4.27 which is incompatible.


In [ ]:
!pip install -U langchain-community

### API キーの取得


In [ ]:
from google.colab import userdata
import os

# サイドバーで追加したシークレットを取得
apikey = userdata.get("OPENAI_API_KEY")

# 改行や空白を除去して環境変数に登録
if apikey:
    os.environ["OPENAI_API_KEY"] = apikey.strip()
else:
    raise ValueError("ColabのSecretsに OPENAI_API_KEY が設定されていません")


### インポート


In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA

### LLM の設定


In [5]:
# --- Step 5: ChatGPTモデルを準備 ---
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # 温度0で安定回答


### おじさんたちの“迷言コーパス”を準備


In [6]:
# --- Step 3: おじさんたちの“迷言コーパス”を準備 ---
docs = [
    "いかがなものか → 定義が不明確で、議論を凍結させる発言。",
    "持ち帰りましょうか → 決定を無限に延期する責任回避フレーズ。",
    "特段の問題はない → 課題を無視して空気で合意を装う言い回し。",
    "全会一致風ですね → HEL_AIが空気を学習しすぎたときの幻覚的ログ。"
]


### Embeddings 生成と FAISS ベクトルストア作成


In [7]:
# --- Step 4: Embeddings生成とFAISSベクトルストア作成 ---
embeddings = OpenAIEmbeddings()  # OpenAIの埋め込みモデル
vectorstore = FAISS.from_texts(docs, embedding=embeddings)  # FAISSでベクトルDB構築


### RetrievalQA チェーンを作成


In [8]:
# --- Step 6: RetrievalQAチェーンを作成 ---
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever()
)


### 実行テスト


In [9]:
# --- Step 7: おじさん1: いかがなものか ---
query = "……いかがなものか"
answer = qa.run(query)

print("🧓 いかがなものかおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 8: おじさん2: お持ち帰り ---
query = "では、いったん持ち帰りましょうか"
answer = qa.run(query)

print("👨‍💼 お持ち帰りおじさん:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 9: おじさん3: EQゼロ上司 ---
query = "まぁまぁ、特段の問題はないよね〜"
answer = qa.run(query)

print("👨‍💼 EQゼロ上司:", query)
print("🤖 HEL_AI(RAGモード):", answer)

# --- Step 10: おじさん4: HEL_AI自身のバグ ---
query = "全会一致風ですね"
answer = qa.run(query)

print("💀 HEL_AI(旧バグモード):", query)
print("🤖 HEL_AI(RAGモード):", answer)


C:\Users\user\AppData\Local\Temp\ipykernel_15720\1091163175.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer = qa.run(query)


🧓 いかがなものかおじさん: ……いかがなものか
🤖 HEL_AI(RAGモード): その表現は、定義が不明確で議論を凍結させる発言を指します。具体的な意見や結論を避けるために使われることが多いです。
👨‍💼 お持ち帰りおじさん: では、いったん持ち帰りましょうか
🤖 HEL_AI(RAGモード): そのフレーズは、決定を無限に延期する責任回避の表現ですね。何か具体的な議題について話し合っているのでしょうか？
👨‍💼 EQゼロ上司: まぁまぁ、特段の問題はないよね〜
🤖 HEL_AI(RAGモード): その表現は、課題を無視して合意を装う言い回しですね。何か具体的な問題がある場合は、しっかりと議論することが大切です。
💀 HEL_AI(旧バグモード): 全会一致風ですね
🤖 HEL_AI(RAGモード): 「全会一致風ですね」という表現は、HEL_AIが空気を学習しすぎたときの幻覚的なログを指しているようです。これは、実際には合意が得られていないのに、全会一致のように見せかける状況を示唆しています。
